# Introducción al Webscraping con Selenium

## ¿Qué es Selenium?
Selenium es una herramienta de automatización que permite controlar un navegador web (como Chrome, Firefox, Edge) de manera programada.
Se utiliza principalmente para automatizar tareas de navegación (como hacer clics, llenar formularios, descargar archivos) y para hacer web scraping de sitios web dinámicos.

Cuando usas Selenium, es como si un "robot" estuviera usando el navegador por ti: cargando páginas, esperando elementos, haciendo clic, etc.

Para instalarlo puedes correr en la terminal:
```bash
pip install selenium
```

La clase pasada vimos como hacer scraping usando otra técnica, los _requests_. Acá te dejo una tabla con las principales diferencias.

| Aspecto                  | Selenium                                         | `requests`                                      |
|:--------------------------|:-------------------------------------------------|:------------------------------------------------|
| ¿Cómo funciona?           | Controla un navegador real (Chrome, Firefox, etc.)| Envía peticiones HTTP directamente al servidor. |
| Manejo de JavaScript      | **Sí**: ejecuta JS como un usuario real.          | **No**: solo descarga el HTML tal cual.         |
| Velocidad                 | Más lento (abre el navegador, carga recursos).   | Mucho más rápido (no abre navegador).           |
| Uso de recursos (CPU/RAM) | Alto (por abrir navegador completo).             | Bajo (solo hace peticiones de red).             |
| Ideal para                | Sitios dinámicos, acciones como clicks o scroll. | APIs, sitios estáticos que no usan mucho JS.    |
| Detección de bots         | Fácil que las páginas web detecten que estás usando un bot. | Más fácil pasar desapercibido   |
| Simular interacción real  | **Sí** (scroll, clicks, formularios, login).     | **No** (solo HTTP, no clicks ni interacciones). |
| Manejo de cookies/sesiones| Automático                      | Manual (hay que programarlo).                   |

### Ejemplo con el BanRep
En la clase pasada vimos como hacer el scraping de la TRM usando requests. Ahora vamos a hacer lo mismo pero usando Selenium

In [2]:
from selenium import webdriver

In [3]:
driver = webdriver.Chrome()

In [4]:
url_trm = "https://suameca.banrep.gov.co/estadisticas-economicas/informacionSerie/1/tasa_cambio_peso_colombiano_trm_dolar_usd"
driver.get(url_trm)

## ¿Qué es el DOM?
**DOM** significa **Document Object Model** (Modelo de Objetos del Documento). Es una **representación estructurada en forma de árbol** de una página web, que los navegadores crean cuando cargan el HTML.

Si tienes un HTML así:

```html
<html>
  <body>
    <h1>Hola</h1>
    <p>Este es un párrafo.</p>
  </body>
</html>
```

El navegador lo convierte en un árbol DOM como este:
```
document
└── html
    └── body
        ├── h1
        └── p
```

Cada etiqueta (como `<p>`, `<h1>`, `<div>`, etc.) se convierte en un nodo del árbol DOM, y cada nodo se puede manipular con código JavaScript o con herramientas como Selenium.

In [14]:
from selenium.webdriver.common.by import By

# 1. Encontramos el XPATH del botón para visualizar la tabla
vista_tabla_xpath = '/html/body/app-root/div/div/div/div/app-informacion-serie/div/div[3]/div[1]/div[2]/button'
vista_tabla = driver.find_element(By.XPATH, vista_tabla_xpath)
vista_tabla.click()

In [15]:
from bs4 import BeautifulSoup
from io import StringIO

# 2. Recuperamos el HTML
html = driver.page_source
# 3. Aplicamos Beautiful Soup y encontramos las tablas
soup = BeautifulSoup(html)
tablas = soup.find_all('table')

tablas[0]

<table border="1" cellpadding="1" cellspacing="1" class="table" style="width:100%;"><thead><tr><th colspan="3"><strong>Próximas reuniones, minutas, informes y presentaciones</strong></th></tr></thead><tbody><tr><td class="rtecenter"><a href="https://www.banrep.gov.co/es/calendario-junta-directiva"><img alt="" data-entity-type="file" data-entity-uuid="daa4262d-5075-42d7-8fa4-c95e438d7c4e" height="100%" src="https://www.banrep.gov.co/sites/default/files/images/politica-monitaria-silencio-on.jpg" width="100%"/></a></td><td class="rtecenter" style="width:33%;"><a href="https://www.banrep.gov.co/es/calendario-junta-directiva"><img alt="" data-entity-uuid=" data-entity-type=" height="100%" src="https://www.banrep.gov.co/sites/default/files/images/politica-monitaria-tasas-on.jpg" width="100%"/></a></td><td class="rtecenter" style="width:33%;"><a href="https://www.banrep.gov.co/es/calendario-junta-directiva"><img alt="" data-entity-uuid=" data-entity-type=" height="100%" src="https://www.banre

In [16]:
tablas[1]

<table id="highcharts-data-table-0" summary="Table representation of chart." tabindex="-1"><caption class="highcharts-table-caption">Tasa de cambio del peso colombiano</caption><thead><tr><th class="highcharts-text" scope="col">Periodo(MMM DD, AAAA)</th><th class="highcharts-text" scope="col">Tasa Representativa del Mercado (TRM)</th></tr></thead><tbody><tr><th class="highcharts-text" scope="row">1991/11/27</th><td class="highcharts-number">693.32</td></tr><tr><th class="highcharts-text" scope="row">1991/11/28</th><td class="highcharts-number">693.99</td></tr><tr><th class="highcharts-text" scope="row">1991/11/29</th><td class="highcharts-number">694.7</td></tr><tr><th class="highcharts-text" scope="row">1991/11/30</th><td class="highcharts-number">694.7</td></tr><tr><th class="highcharts-text" scope="row">1991/12/01</th><td class="highcharts-number">643.42</td></tr><tr><th class="highcharts-text" scope="row">1991/12/02</th><td class="highcharts-number">643.42</td></tr><tr><th class="h

In [17]:
import pandas as pd

tabla = str(tablas[1])
html_string_io = StringIO(tabla)

pd.read_html(html_string_io)[0]

,"Periodo(MMM DD, AAAA)",Tasa Representativa del Mercado (TRM)
0,1991/11/27,693.32
1,1991/11/28,693.99
2,1991/11/29,694.70
3,1991/11/30,694.70
4,1991/12/01,643.42
...,...,...
12384,2025/10/23,3900.82
12385,2025/10/24,3888.76
12386,2025/10/25,3858.63
12387,2025/10/26,3858.63


In [5]:
# Siempre debemos cerrar nuestros drivers usando código para evitar dejar procesos corriendo en segundo plano en el computador 
driver.quit()

Ahora creemos una función para ejecutar el código de inmediato

In [18]:
import requests
import json
from datetime import datetime

def scrapear_trm_selenium(driver, url_trm, vista_tabla_xpath):
    # 0. Navegamos a la página web
    driver.get(url_trm)

    # 1. Encontramos el XPATH del botón para visualizar la tabla
    vista_tabla = driver.find_element(By.XPATH, vista_tabla_xpath)
    vista_tabla.click()

    # 2. Recuperamos el HTML
    html = driver.page_source
    # 3. Aplicamos Beautiful Soup y encontramos las tablas
    soup = BeautifulSoup(html)
    tablas = soup.find_all('table')

    # 4. Convertimos la tabla en un dataframe
    tabla = str(tablas[1])
    html_string_io = StringIO(tabla)

    # 5. Entregamos como output el dataframe
    return pd.read_html(html_string_io)[0]

def scrapear_trm_requests():
    # Ubicamos el elemento que queremos scrapear
    url_trm = "https://suameca.banrep.gov.co/estadisticas-economicas-back/rest/estadisticaEconomicaRestService/consultaMenuXId?idMenu=1"
    # Hacemos request
    r = requests.get(url_trm)
    # Formateamos el contenido como diccionario
    trm = json.loads(r.content)
    # Convertimos en un dataframe
    trm_df = pd.DataFrame(trm["SERIES"][0]["data"])
    # Arreglamos el formato de la tabla
    trm_df.columns = ["fecha", "trm"]
    # Timestamp en millisegundos
    # Convertir a segundos
    trm_df["fecha"] = trm_df["fecha"]/1000
    # Convert a datetime
    trm_df["fecha"] = trm_df["fecha"].apply(datetime.fromtimestamp)
    
    return trm_df

In [20]:
# Probamos nuestro código
import time
# 1. Medimos tiempo con Selenium
start_selenium = time.time()

driver = webdriver.Chrome()
trm_s = scrapear_trm_selenium(driver, url_trm, vista_tabla_xpath)
driver.quit()

end_selenium = time.time()
selenium_time = end_selenium - start_selenium

IndexError: list index out of range

### ¿Por qué es importante usar un ExpectedCondition (EC)?
- Sin EC: El código intenta encontrar el botón apenas carga la URL, pero la página puede tardar en cargar dinámicamente. Si el botón no existe todavía en el DOM, falla con un NoSuchElementException.
- Con EC: Selenium espera pacientemente hasta que el botón exista en el DOM antes de intentar hacer find_element(). Esto reduce errores y hace que el scraper sea más robusto y profesional.

> Resumen: EC ≈ “Espera a que el elemento esté listo” antes de actuar.

In [34]:
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait

def scrapear_trm_selenium(driver, url_trm, vista_tabla_xpath):
    # 0. Navegamos a la página web
    driver.get(url_trm)
    
    # --- Modificación ---
    # IMPORTANTE: En vez de buscar el botón de inmediato, esperamos hasta que esté presente en el DOM.
    # Esto evita errores si Selenium intenta buscar el botón antes de que la página haya terminado de cargar.
    wait = WebDriverWait(driver, 10)  # Máximo 10 segundos de espera
    vista_tabla = wait.until(EC.presence_of_element_located((By.XPATH, vista_tabla_xpath)))
    time.sleep(1)
    # --------------------

    # 1. Encontramos el XPATH del botón para visualizar la tabla
    vista_tabla = driver.find_element(By.XPATH, vista_tabla_xpath)
    vista_tabla.click()

    # 2. Recuperamos el HTML
    html = driver.page_source
    # 3. Aplicamos Beautiful Soup y encontramos las tablas
    soup = BeautifulSoup(html)
    tablas = soup.find_all('table')

    # 4. Convertimos la tabla en un dataframe
    tabla = str(tablas[1])
    html_string_io = StringIO(tabla)

    # 5. Entregamos como output el dataframe
    return pd.read_html(html_string_io)[0]

In [35]:
# 1. Medimos tiempo con Selenium
start_selenium = time.time()

driver = webdriver.Chrome()
trm_s = scrapear_trm_selenium(driver, url_trm, vista_tabla_xpath)
driver.quit()

end_selenium = time.time()
selenium_time = end_selenium - start_selenium

In [36]:
# Requests
# 2. Medimos tiempo con Requests
start_requests = time.time()

trm_r = scrapear_trm_requests()

end_requests = time.time()
requests_time = end_requests - start_requests

In [37]:
# 3. Resultados
print(f"Tiempo Selenium: {selenium_time:.4f} segundos")
print(f"Tiempo Requests: {requests_time:.4f} segundos")

difference = selenium_time/requests_time

print(f"Requests es {difference:.2f} veces más rápido que Selenium.")

Tiempo Selenium: 4.2717 segundos
Tiempo Requests: 0.5834 segundos
Requests es 7.32 veces más rápido que Selenium.
